# SO(3) benchmarking and profiling

Runs the generic benchmark helper with an explicit SO(3) backend, then profiles the same benchmark call with the standard-library `cProfile` profiler.


In [9]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys

start = Path.cwd().resolve()
candidates = [start, *start.parents]
try:
    candidates.extend(path for path in start.iterdir() if path.is_dir())
except OSError:
    pass

PROJECT_ROOT = next(
    (
        path
        for path in candidates
        if (path / "pyproject.toml").exists()
        and (path / "src" / "equivariant_polynomials").is_dir()
        and (path / "benchmarks").is_dir()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError(
        "Could not find the project root. Start Jupyter from the project directory "
        "or from its notebooks directory."
    )

for import_path in (PROJECT_ROOT, PROJECT_ROOT / "src"):
    import_path = str(import_path)
    if import_path not in sys.path:
        sys.path.insert(0, import_path)

print(f"Project root: {PROJECT_ROOT}")


Project root: C:\Users\micha\OneDrive\Desktop\Oden\Notes\Equivariant Polynomial Tensor Functions\Python\project


In [10]:
import cProfile
import io
import pstats

from IPython.display import Markdown, display

from benchmarks import benchmark, format_benchmark_markdown
from equivariant_polynomials.core import suggest_prime_modulus
from equivariant_polynomials.groups.so3 import SO3RepresentationTheory, hilbert_series_so3


In [11]:
random_seed = 12345
input_irreps = (2, 3)
input_multiplicities = (2, 1)
output_irreps = (4,)
max_degree = 7
invariant_irrep = 0
modulus = suggest_prime_modulus(max_degree)
profile_sort = "cumtime"
profile_rows = 25

benchmark_config = {
    "hilbert_series": hilbert_series_so3,
    "input_irreps": input_irreps,
    "input_multiplicities": input_multiplicities,
    "max_degree": max_degree,
    "invariant_irrep": invariant_irrep,
    "random_seed": random_seed,
    "modulus": modulus,
}


## Benchmark


In [12]:
theory = SO3RepresentationTheory()
summaries = []

for output_irrep in output_irreps:
    print(f"\nRun: output_irrep={output_irrep}, max_degree={max_degree}")
    summaries.append(
        benchmark(
            theory=theory,
            output_irrep=output_irrep,
            verbose=True,
            **benchmark_config,
        )
    )

summaries = tuple(summaries)
display(Markdown(format_benchmark_markdown(summaries)))



Run: output_irrep=4, max_degree=7
algebra tree: 0.478 s
algebra generators: 0.745 s
module tree: 2.630 s
module generators: 8.054 s
total: 11.910 s


### Generator counts

| output irrep | candidate algebra | independent algebra | candidate module | independent module | checks |
| --- | --- | --- | --- | --- | --- |
| 4 | `(1, 0, 4, 7, 24, 54, 156, 340)` | `(1, 0, 4, 7, 14, 26, 52, 68)` | `(0, 0, 6, 21, 87, 273, 807, 2109)` | `(0, 0, 6, 21, 63, 147, 264, 290)` | passed |

### Timings (seconds)

| output irrep | invariant tree | algebra basis | covariant tree | module basis | total |
| --- | --- | --- | --- | --- | --- |
| 4 | 0.478 | 0.745 | 2.630 | 8.054 | 11.910 |

## cProfile


In [13]:
profiled_output_irrep = output_irreps[0]
profiler = cProfile.Profile()
profile_summary = profiler.runcall(
    benchmark,
    theory=SO3RepresentationTheory(),
    output_irrep=profiled_output_irrep,
    verbose=False,
    **benchmark_config,
)

stats_stream = io.StringIO()
stats = pstats.Stats(profiler, stream=stats_stream)
stats.strip_dirs().sort_stats(profile_sort).print_stats(profile_rows)
print(stats_stream.getvalue())

profile_summary


         3689789 function calls (3453302 primitive calls) in 13.021 seconds

   Ordered by: cumulative time
   List reduced from 265 to 25 due to restriction <25>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        4    0.005    0.001   12.488    3.122 selectors.py:310(select)
        4    0.041    0.010    6.900    1.725 selectors.py:304(_select)
    36052    0.039    0.000    5.018    0.000 evaluators.py:174(evaluate_tensor_train)
       18    0.000    0.000    4.698    0.261 isotypic.py:148(<genexpr>)
       16    0.007    0.000    4.693    0.293 isotypic.py:22(build_isotypic_data_tree)
        2    0.000    0.000    4.642    2.321 isotypic.py:133(build_isotypic_data_trees_by_degree)
     1938    0.002    0.000    4.586    0.002 isotypic.py:88(<genexpr>)
      138    0.005    0.000    4.584    0.033 bases.py:329(_schur_functor_basis)
        4    4.263    1.066    4.400    1.100 {built-in method select.select}
    19460    0.866    0.000    4.162    0.00

{'scenario': {'input_irreps': (2, 3),
  'input_multiplicities': (2, 1),
  'output_irrep': 4,
  'invariant_irrep': 0,
  'max_degree': 7,
  'random_seed': 12345,
  'modulus': 2521},
 'algebra': {'candidate_dimensions': (1, 0, 4, 7, 24, 54, 156, 340),
  'hilbert_dimensions': (1, 0, 4, 7, 24, 54, 156, 340),
  'matches_hilbert': True,
  'generator_counts': (1, 0, 4, 7, 14, 26, 52, 68),
  'tree_seconds': 0.8160317000001669,
  'generator_seconds': 0.9039749000221491},
 'module': {'candidate_dimensions': (0, 0, 6, 21, 87, 273, 807, 2109),
  'hilbert_dimensions': (0, 0, 6, 21, 87, 273, 807, 2109),
  'matches_hilbert': True,
  'generator_counts': (0, 0, 6, 21, 63, 147, 264, 290),
  'tree_seconds': 3.8669314000289887,
  'generator_seconds': 7.4387096999562345},
 'total_seconds': 13.029195199953392}